# scVI balanced-data baselines

All four arms are trained and evaluated with `bash/run_scvi.sh`
(stages `train`, `eval-recon`, `eval-class`):

| Arm | Results |
|---|---|
| Standard, AR | `result/test/scVI-{reconstruction,classification}-evals/` |
| Standard + Class Balance, Standard + Geometric Sketching | `result/test/scVI-{reconstruction,classification}-evals-balanced-data/` |

- Every expected result file (4 arms × 7 atlas sizes × 5 seeds) is loaded by name; missing or
  unexpected files stop the notebook.
- Figures and p-value tables are written to `FIGURE_DIR` and `PVALUE_DIR`.
- The classification section runs only when all 420 classification CSVs exist
  (`CLASSIFICATION_READY`); the p-value tables then use the classification metrics and the
  reconstruction correlation (otherwise the reconstruction correlation only).

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.lines import Line2D
from scipy import stats

RESULT_DIR = Path('../result/test')
RECON_DIRS = {
    'main': RESULT_DIR / 'scVI-reconstruction-evals',
    'balanced': RESULT_DIR / 'scVI-reconstruction-evals-balanced-data',
}
CLASS_DIRS = {
    'main': RESULT_DIR / 'scVI-classification-evals',
    'balanced': RESULT_DIR / 'scVI-classification-evals-balanced-data',
}
FIGURE_DIR = Path('../result/figures/sctab/scvi-balanced')
PVALUE_DIR = FIGURE_DIR / 'pvalues'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PVALUE_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44, 45, 46]
ATLAS_COUNTS = [0, 1, 10, 100, 1000, 10000, 50000]
DATASET_ORDER = ['Heart', 'Kidney', 'Neurons']
# run settings shared by all arms (part of the result file names)
LATENT_DIM, ALPHA, BINS, EPOCHS = 64, '0.0001', '100', 300

balancing_order = [
    'Standard',
    'Standard + Class Balance',
    'Standard + Geometric Sketching',
    'AR',
]
# result directory and file-name suffix of each arm (the eval scripts write
# 'Original Combination' / '' for the non-balanced Standard and AR models)
ARM_FILES = {
    'Standard': ('main', 'F', 'Original Combination', ''),
    'AR': ('main', 'T', 'Original Combination', ''),
    'Standard + Class Balance': ('balanced', 'F', 'class_balancing', 'class_balancing'),
    'Standard + Geometric Sketching': ('balanced', 'F', 'geometric_sketching', 'geometric_sketching'),
}

sns.set_theme(style='whitegrid')

In [ ]:
def recon_file(method, seed, atlas):
    where, ar, recon_suffix, _ = ARM_FILES[method]
    return RECON_DIRS[where] / (
        f'Reconstruction_seed_{seed}_ARtype_{ar}_latent_dim_{LATENT_DIM}_'
        f'Atlas_cell_count_{atlas}_alpha_{ALPHA}_bins_{BINS}_epoch_{EPOCHS}_'
        f'{recon_suffix}_bloodbase_eval.pkl'
    )


def class_file(method, dataset, seed, atlas):
    where, ar, _, class_suffix = ARM_FILES[method]
    return CLASS_DIRS[where] / (
        f'zero_shot_classification_metrics_scVI_{dataset}_seed_{seed}_ARtype_{ar}_'
        f'latent_dim_{LATENT_DIM}_Atlas_cell_count_{atlas}_alpha_{ALPHA}_bins_{BINS}_'
        f'epoch_{EPOCHS}_{class_suffix}.csv'
    )


def check_files(expected, dirs, pattern):
    # every expected file exists and the result dirs hold no other result file
    expected = set(expected)
    found = {p for d in dirs.values() if d.exists() for p in d.glob(pattern)}
    missing = sorted(expected - found)
    unexpected = sorted(found - expected)
    print(f'{len(expected)} expected, {len(found & expected)} found, '
          f'{len(missing)} missing, {len(unexpected)} unexpected')
    for p in unexpected:
        print('  unexpected:', p)
    return missing, unexpected

## Reconstruction

In [ ]:
RECON_DATASETS = {
    'scTab_Heart_reconstruction': 'Heart',
    'scTab_Kidney_reconstruction': 'Kidney',
    'Neurons_reconstruction': 'Neurons',
}

recon_files = {
    (method, seed, atlas): recon_file(method, seed, atlas)
    for method in balancing_order for seed in SEEDS for atlas in ATLAS_COUNTS
}
missing, unexpected = check_files(recon_files.values(), RECON_DIRS, 'Reconstruction_seed_*.pkl')
assert not missing and not unexpected, missing[:5]

rows = []
for (method, seed, atlas), path in recon_files.items():
    with open(path, 'rb') as f:
        evals = pickle.load(f)
    assert set(evals) == set(RECON_DATASETS), (path, list(evals))
    for key, cells in evals.items():
        for metric in ['correlation', 'r2']:
            rows.append({
                'dataset': RECON_DATASETS[key],
                'metric': metric,
                'value': cells[metric].mean(),
                'Model': 'AR' if method == 'AR' else 'Standard',
                'Balancing method': method,
                'Atlas_cell_count': atlas,
                'seed': seed,
                'n_cells': len(cells),
            })
recon_df = pd.DataFrame(rows)

seed_counts = recon_df.groupby('seed').size()
print(seed_counts)
assert set(seed_counts.index) == set(SEEDS)
assert (seed_counts == 168).all(), (
    f'Expected 168 rows per seed (4 methods x 3 datasets x 2 metrics x 7 atlas sizes), '
    f'got {seed_counts.to_dict()}'
)
assert recon_df['value'].notna().all()
# same test cells for every model
print(recon_df.groupby('dataset')['n_cells'].unique())
assert (recon_df.groupby('dataset')['n_cells'].nunique() == 1).all()

In [ ]:
# mean (sd) over the five seeds
recon_summary = (
    recon_df[recon_df['metric'] == 'correlation']
    .groupby(['dataset', 'Balancing method', 'Atlas_cell_count'])['value']
    .agg(['mean', 'std', 'count'])
    .round(4)
)
recon_summary.to_csv(FIGURE_DIR / 'reconstruction_correlation_summary.csv')
display(recon_summary.unstack('Atlas_cell_count')['mean'].reindex(balancing_order, level='Balancing method'))

In [ ]:
# Set all lines to darkgray except AR, which is brown
palette = {
    'Standard': "darkgray",
    'Standard + Class Balance': "darkgray",
    'Standard + Geometric Sketching': "darkgray",
    'AR': "brown"
}

markers = {
    'Standard': 'o',
    'Standard + Class Balance': 's',
    'Standard + Geometric Sketching': '^',
    'AR': 'o'
}

# Set dashes for line styles
dashes = {
    'Standard': '',
    'Standard + Class Balance': (2, 2),
    'Standard + Geometric Sketching': (4, 2),
    'AR': ''
}


def plot_metric(data, dataset, metric, ylabel, file_name):
    # x on a log scale (so atlas count 0 is not shown),
    # seaborn's default 95% bootstrap CI over seeds, y from 0 to 1
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    g = sns.lineplot(
        data=data[(data['dataset'] == dataset) & (data['metric'] == metric)],
        x="Atlas_cell_count",
        y="value",
        hue="Balancing method",
        style="Balancing method",
        hue_order=balancing_order,
        style_order=balancing_order,
        markers=markers,
        dashes=dashes,
        palette=palette,
        legend=False,  # We'll handle legend manually
        ax=ax,
    )
    g.set_xlabel('')
    g.set_ylabel(ylabel, fontsize=15)
    g.set_title(dataset.capitalize(), fontsize=18)
    g.tick_params(axis='both', which='major', labelsize=15)
    g.tick_params(axis='both', which='minor', labelsize=15)
    g.set_xscale("log", base=10)

    # --- Custom legend just for Balancing method, matching color and dashes/markers ---
    legend_elements = [
        Line2D(
            [0], [0],
            color=palette[method],
            marker=markers[method],
            linestyle='-' if dashes[method] == '' else (0, dashes[method]),
            label=method,
            markersize=8,
            linewidth=2
        )
        for method in balancing_order
    ]
    g.legend(
        handles=legend_elements,
        title=None,
        loc='upper left',
        fontsize=15,
        bbox_to_anchor=(1, 1)
    )

    g.set(ylim=(0, 1))
    plt.savefig(FIGURE_DIR / file_name, dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
for dataset in DATASET_ORDER:
    for metric in ['correlation']:
        plot_metric(recon_df, dataset, metric, metric.capitalize(),
                    f'reconstruction_{dataset}_{metric}.png')

## Classification

Runs only when all 4 arms × 3 datasets × 7 atlas sizes × 5 seeds = 420 CSVs exist.

In [ ]:
class_files = {
    (method, dataset, seed, atlas): class_file(method, dataset, seed, atlas)
    for method in balancing_order for dataset in DATASET_ORDER
    for seed in SEEDS for atlas in ATLAS_COUNTS
}
missing, unexpected = check_files(
    class_files.values(), CLASS_DIRS, 'zero_shot_classification_metrics_scVI_*.csv')
assert not unexpected
CLASSIFICATION_READY = not missing
if not CLASSIFICATION_READY:
    print(f'Classification results incomplete ({len(missing)} files missing): '
          'classification plots and tables are skipped. Rerun the notebook when the evals are done.')

In [ ]:
CLASSIFICATION_METRICS = [
    'Accuracy', 'Precision', 'Recall', 'Micro F1 Score', 'Macro F1 Score'
]
CLASS_COLUMNS = {
    'accuracy': 'Accuracy', 'precision': 'Precision', 'recall': 'Recall',
    'micro_f1': 'Micro F1 Score', 'macro_f1': 'Macro F1 Score',
}

plot_df = pd.DataFrame(columns=['dataset', 'metric', 'value', 'Model', 'Balancing method',
                                'Atlas_cell_count', 'seed'])
if CLASSIFICATION_READY:
    rows = []
    for (method, dataset, seed, atlas), path in class_files.items():
        metrics_df = pd.read_csv(path, index_col=0)
        assert len(metrics_df) == 1, path
        row = metrics_df.iloc[0]
        # the file name and the columns written by the eval script agree
        assert (row['dataset'], int(row['seed']), int(row['Atlas_cell_count'])) == (dataset, seed, atlas), path
        assert row['ARtype'] == ARM_FILES[method][1], path
        for column, metric in CLASS_COLUMNS.items():
            rows.append({
                'dataset': dataset,
                'metric': metric,
                'value': row[column],
                'Model': 'AR' if method == 'AR' else 'Standard',
                'Balancing method': method,
                'Atlas_cell_count': atlas,
                'seed': seed,
            })
    plot_df = pd.DataFrame(rows)

    seed_counts = plot_df.groupby('seed').size()
    print(seed_counts)
    assert set(seed_counts.index) == set(SEEDS)
    assert (seed_counts == 420).all(), (
        f'Expected 420 rows per seed (4 methods x 3 datasets x 7 atlas sizes x 5 metrics), '
        f'got {seed_counts.to_dict()}'
    )
    assert plot_df['value'].notna().all()

In [ ]:
if CLASSIFICATION_READY:
    class_summary = (
        plot_df.groupby(['dataset', 'metric', 'Balancing method', 'Atlas_cell_count'])['value']
        .agg(['mean', 'std', 'count'])
        .round(4)
    )
    class_summary.to_csv(FIGURE_DIR / 'classification_summary.csv')
    display(
        class_summary.xs('Accuracy', level='metric')
        .unstack('Atlas_cell_count')['mean']
        .reindex(balancing_order, level='Balancing method')
    )

In [ ]:
if CLASSIFICATION_READY:
    for dataset in DATASET_ORDER:
        for metric in CLASSIFICATION_METRICS:
            plot_metric(plot_df, dataset, metric, metric,
                        f'classification_{dataset}_{metric}.png')

## Supplementary p-value tables

Seed-matched two-sided **paired t-tests** across five random seeds, comparing **AR** to each
method at every atlas-cell count and evaluation dataset (AR vs Standard, AR vs class balancing,
AR vs geometric sketching): 9 tables. Bold = $P<0.05$; dagger = $P<0.05$ and AR has the higher mean.

Uses the classification metrics when `CLASSIFICATION_READY`, and the reconstruction
correlation. Pairwise p-values are nominal and uncorrected.

In [ ]:
TABLE_METRICS = (CLASSIFICATION_METRICS if CLASSIFICATION_READY else []) + ['Correlation']
TABLE_METRICS_DISPLAY_ORDER = [
    m for m in ['Accuracy', 'Macro F1 Score', 'Micro F1 Score', 'Precision', 'Recall', 'Correlation']
    if m in TABLE_METRICS
]
# row labels of the LaTeX tables (Correlation = mean per-cell Pearson correlation)
METRIC_LABELS = {
    'Correlation': 'Correlation',
}
BASELINE_COMPARISONS = {
    'standard': 'Standard',
    'class_balancing': 'Standard + Class Balance',
    'geometric_sketching': 'Standard + Geometric Sketching',
}
COMPARISON_TITLES = {
    'standard': 'AR vs Standard',
    'class_balancing': 'AR vs Class Balancing',
    'geometric_sketching': 'AR vs Geometric Sketching',
}
BASELINE_CAPTION_NAMES = {
    'standard': 'standard scVI',
    'class_balancing': 'class balancing',
    'geometric_sketching': 'geometric sketching',
}
TEST_CAPTION_PHRASES = {
    'paired_ttest': 'paired t-tests',
}
STAT_TESTS = {
    'paired_ttest': 'Paired t-test',
}
if not CLASSIFICATION_READY:
    print('Classification results incomplete: p-value tables contain the reconstruction correlation only.')


def build_table_caption(test_key, comparison_key, dataset):
    test_phrase = TEST_CAPTION_PHRASES[test_key]
    baseline_name = BASELINE_CAPTION_NAMES[comparison_key]
    dataset_name = dataset.lower()
    return (
        f'The p-values for two-sided seed-matched {test_phrase} on evaluation metrics '
        f'for AR and {baseline_name} tested on the {dataset_name} dataset. '
        f'Bold indicates statistical significance as determined by $P<0.05$. '
        f'A dagger ($^{{\\dagger}}$) marks the significant values where AR has the higher mean. '
        f'Results are based on five replicates.'
    )


def format_pval_latex(val, ar_outperforms=False):
    if pd.isna(val):
        return ''
    formatted = f'{val:.2e}'
    if val < 0.05:
        mark = r'$^{\dagger}$' if ar_outperforms else ''
        return rf'\textbf{{{formatted}}}{mark}'
    return rf'\cellcolor{{white}}{{{formatted}}}'


def latex_pvalue_table(pivot_pvalues, pivot_winners, caption, label):
    atlas_cols = pivot_pvalues.columns.tolist()
    col_spec = 'l' + 'r' * len(atlas_cols)
    header = r'\# Atlas Cells & ' + ' & '.join(str(c) for c in atlas_cols) + r' \\'
    lines = [
        r'\begin{table*}[ht]',
        rf'\caption{{{caption}}}',
        r'\centering',
        rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{{col_spec}}}',
        r'\toprule',
        header,
        r'\midrule',
    ]
    for metric in TABLE_METRICS_DISPLAY_ORDER:
        if metric == 'Correlation' and len(TABLE_METRICS_DISPLAY_ORDER) > 1:
            lines.append(r'\midrule')
        row_label = METRIC_LABELS.get(metric, metric)
        row_cells = [row_label]
        for col in atlas_cols:
            p_val = pivot_pvalues.loc[metric, col]
            ar_outperforms = pivot_winners.loc[metric, col] == 'AR'
            row_cells.append(format_pval_latex(p_val, ar_outperforms=ar_outperforms))
        lines.append(' & '.join(row_cells) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    return '\n'.join(lines)


def _paired_pvalue(ar_values, baseline_values, test_key):
    if test_key == 'paired_ttest':
        return stats.ttest_rel(ar_values, baseline_values).pvalue
    raise ValueError(f'Unknown test: {test_key}')


def compute_ar_vs_baseline_pvalues(
    df,
    baseline_label,
    test_key='paired_ttest',
    metrics=None,
    model_col='Model',
    balancing_col='Balancing method',
    seed_col='seed',
    atlas_order=None,
):
    # Return long-form seed-matched p-values for AR vs one baseline method.
    metrics = metrics or TABLE_METRICS
    atlas_order = atlas_order or sorted(df['Atlas_cell_count'].unique())
    records = []
    for dataset in DATASET_ORDER:
        for metric in metrics:
            for atlas_cell_count in atlas_order:
                ar_df = df[
                    (df['dataset'] == dataset)
                    & (df['metric'] == metric)
                    & (df['Atlas_cell_count'] == atlas_cell_count)
                    & (df[model_col] == 'AR')
                    & (df[balancing_col] == 'AR')
                ][[seed_col, 'value']]
                baseline_df = df[
                    (df['dataset'] == dataset)
                    & (df['metric'] == metric)
                    & (df['Atlas_cell_count'] == atlas_cell_count)
                    & (df[model_col] == 'Standard')
                    & (df[balancing_col] == baseline_label)
                ][[seed_col, 'value']]
                merged = ar_df.merge(
                    baseline_df,
                    on=seed_col,
                    suffixes=('_ar', '_bl'),
                ).sort_values(seed_col)
                # all five seeds must be paired
                assert len(merged) == len(SEEDS), (dataset, metric, atlas_cell_count, baseline_label)
                p_value = _paired_pvalue(merged['value_ar'], merged['value_bl'], test_key)
                ar_mean = merged['value_ar'].mean()
                baseline_mean = merged['value_bl'].mean()
                records.append(
                    {
                        'dataset': dataset,
                        'metric': metric,
                        'Atlas_cell_count': atlas_cell_count,
                        'p_value': p_value,
                        'ar_mean': ar_mean,
                        'baseline_mean': baseline_mean,
                        'n_pairs': len(merged),
                        'test': test_key,
                        'higher_method': 'AR' if ar_mean > baseline_mean else baseline_label,
                    }
                )
    return pd.DataFrame(records)


def pvalue_table(df, dataset, metrics=None, atlas_order=None, value_col='p_value'):
    subset = df[df['dataset'] == dataset].copy()
    metrics = metrics or TABLE_METRICS
    atlas_order = atlas_order or sorted(subset['Atlas_cell_count'].unique())
    pivot = subset.pivot(index='metric', columns='Atlas_cell_count', values=value_col)
    pivot = pivot.reindex(index=metrics, columns=atlas_order)
    return pivot

In [ ]:
reconstruction_for_pvalues = recon_df[recon_df['metric'] == 'correlation'].copy()
reconstruction_for_pvalues['metric'] = 'Correlation'

pvalue_input_cols = [
    'dataset', 'metric', 'value', 'Model', 'Balancing method', 'Atlas_cell_count', 'seed'
]
pvalue_input_df = pd.concat(
    ([plot_df[pvalue_input_cols]] if CLASSIFICATION_READY else [])
    + [reconstruction_for_pvalues[pvalue_input_cols]],
    ignore_index=True,
)
pvalue_input_df['Atlas_cell_count'] = pvalue_input_df['Atlas_cell_count'].astype(int)
pvalue_input_df['seed'] = pvalue_input_df['seed'].astype(int)
pvalue_input_df['value'] = pvalue_input_df['value'].astype(float)

COMBINED_ATLAS_ORDER = sorted(pvalue_input_df['Atlas_cell_count'].unique())

pvalue_tables = {test_key: {} for test_key in STAT_TESTS}
for test_key in STAT_TESTS:
    for comparison_key, baseline_label in BASELINE_COMPARISONS.items():
        pvalue_tables[test_key][comparison_key] = compute_ar_vs_baseline_pvalues(
            pvalue_input_df,
            baseline_label,
            test_key=test_key,
            metrics=TABLE_METRICS,
            atlas_order=COMBINED_ATLAS_ORDER,
        )
        pvalue_tables[test_key][comparison_key].to_csv(
            PVALUE_DIR / f'pvalues_{test_key}_AR_vs_{comparison_key}.csv',
            index=False,
        )

print('Saved CSV summaries to', PVALUE_DIR)
print('Metrics:', TABLE_METRICS)
print('Rows per test/comparison:')
for test_key in STAT_TESTS:
    print(test_key, {k: len(v) for k, v in pvalue_tables[test_key].items()})

In [ ]:
latex_by_test = {test_key: [] for test_key in STAT_TESTS}
table_manifest = []

for test_key, test_label in STAT_TESTS.items():
    print('\n' + '#' * 100)
    print(f'# {test_label}')
    print('#' * 100)
    for comparison_key, baseline_label in BASELINE_COMPARISONS.items():
        comparison_df = pvalue_tables[test_key][comparison_key]
        for dataset in DATASET_ORDER:
            pivot = pvalue_table(
                comparison_df,
                dataset,
                metrics=TABLE_METRICS,
                atlas_order=COMBINED_ATLAS_ORDER,
            )
            winner_pivot = pvalue_table(
                comparison_df,
                dataset,
                metrics=TABLE_METRICS,
                atlas_order=COMBINED_ATLAS_ORDER,
                value_col='higher_method',
            )
            safe_dataset = dataset.lower()
            label = f'tab:supp_pvalues_{test_key}_{comparison_key}_{safe_dataset}'
            caption = build_table_caption(test_key, comparison_key, dataset)
            latex = latex_pvalue_table(pivot, winner_pivot, caption=caption, label=label)
            latex_by_test[test_key].append(latex)
            table_manifest.append(
                {
                    'test': test_key,
                    'comparison': comparison_key,
                    'dataset': dataset,
                    'label': label,
                    'caption': caption,
                    'metrics': ', '.join(TABLE_METRICS),
                }
            )
            print('=' * 80)
            print(caption)
            display(pivot)
            print(latex)

manifest_path = PVALUE_DIR / 'supplementary_pvalue_table_manifest.csv'
pd.DataFrame(table_manifest).to_csv(manifest_path, index=False)

for test_key in STAT_TESTS:
    combined_latex_path = PVALUE_DIR / f'supplementary_pvalue_tables_{test_key}.tex'
    combined_latex_path.write_text('\n\n'.join(latex_by_test[test_key]))
    print('\nSaved', combined_latex_path)

print('Saved manifest to', manifest_path)

if CLASSIFICATION_READY:
    # Quick view: Neurons accuracy, AR vs class balancing
    for test_key in STAT_TESTS:
        print(f'\nNeurons accuracy ({test_key}, class balancing):')
        display(
            pvalue_tables[test_key]['class_balancing']
            .query("dataset == 'Neurons' and metric == 'Accuracy'")
            .sort_values('Atlas_cell_count')
            [['Atlas_cell_count', 'p_value', 'ar_mean', 'baseline_mean', 'n_pairs', 'higher_method']]
        )

In [ ]:
# Optional heatmaps of the paired t-test tables
for test_key, test_label in STAT_TESTS.items():
    for comparison_key, comparison_title in COMPARISON_TITLES.items():
        comparison_df = pvalue_tables[test_key][comparison_key]
        for dataset in DATASET_ORDER:
            pivot_df = pvalue_table(
                comparison_df,
                dataset,
                metrics=TABLE_METRICS,
                atlas_order=COMBINED_ATLAS_ORDER,
            )
            plt.figure(figsize=(10, max(2, 0.6 * len(TABLE_METRICS) + 1)))
            sns.heatmap(
                pivot_df,
                annot=True,
                fmt='.1e',
                cmap='RdYlGn_r',
                vmin=0,
                vmax=0.1,
                cbar_kws={'label': 'P-Value'},
            )
            plt.title(f'{test_label}: {comparison_title} ({dataset})')
            plt.xlabel('Atlas Cell Count')
            plt.ylabel('Metric')
            plt.tight_layout()
            plt.show()